# 50 X/Y Probe — Balanced Writing-Type Independent Validation v5

[**Open in Google Colab — v5 (new notebook)**](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/experiments/2026-11/1124_probe_50pairs/qwen_probe_50pairs_balanced_validation_v5.ipynb)

This is a **new notebook**; v1 and v2 are preserved unchanged. It tests 50 fixed X/Y pairs by matching four explanation types (**structure, action, situation, function**) in both train and held-out sets, with two sentences per type per split. It compares held-out AUC against original v1.

**Change to fix the repeated error:** Generate and verify **one style and two sentences at a time** rather than requiring four categories in one API response. JSON Schema enforces exactly two strings; each response is checked, logged and retried if necessary. Every concept is saved immutably for resumption. A failed response is **not silently included** in the dataset.

**Setup:** Colab GPU runtime, Secrets `OPENAI_API_KEY` and `GITHUB_RESULTS_TOKEN` (repository Contents read/write), then **Runtime → Run all**. The code is published but API/GPU execution has not yet been verified.

The underlying concept pairs, Qwen revision, layers, sample count, Probe estimator and descriptive thresholds are identical to v2. This is an exploratory follow-up, not an independent proof of concept activation or suppression.

In [1]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" "openai>=1.70,<3" "statsmodels>=0.14,<1" pandas numpy requests tqdm matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, re, io, json, math, time, base64, hashlib, random, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
import numpy as np, pandas as pd, requests, torch
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from google.colab import userdata

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
REPO = "ryouy/GraduationThesis"
SOURCE_CSV = "docs/concept_pair_candidates_50_v0_1.csv"
ROOT = "experiments/2026-11/1124_probe_50pairs"
RUN_ID = "20261124_probe50_balanced_v5"
DEST = f"{ROOT}/results/{RUN_ID}"
WORK = Path("/content/qwen_probe_50pairs") / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
API_MODEL = "gpt-4o-mini"   # Change only with a NEW RUN_ID.
LAYERS = [22, 23, 24]
PER_CLASS = 8
SHUFFLES = 499
BATCH_SIZE = 24
PROBE_C = 0.1
AUC_MIN = 0.80
FDR_MAX = 0.05

def secret(name):
    try: value = userdata.get(name)
    except Exception: value = None
    if not value: raise RuntimeError(f"Set Colab Secret {name} and grant notebook access")
    return value

OPENAI = OpenAI(api_key=secret("OPENAI_API_KEY"))
GH_TOKEN = secret("GITHUB_RESULTS_TOKEN")
gh = requests.Session()
gh.headers.update({"Authorization": f"Bearer {GH_TOKEN}",
                   "Accept": "application/vnd.github+json",
                   "X-GitHub-Api-Version": "2022-11-28"})
API_ROOT = f"https://api.github.com/repos/{REPO}/contents/"

def git_meta(p):
    r = gh.get(API_ROOT+p, timeout=90)
    if r.status_code == 404: return None
    r.raise_for_status()
    return r.json()

def git_get(p):
    info = git_meta(p)
    if info is None: return None
    r = gh.get(info["download_url"], timeout=180); r.raise_for_status()
    return r.content

def blob_sha(content):
    return hashlib.sha1(b"blob "+str(len(content)).encode()+b"\0"+content).hexdigest()

def git_put_immutable(p, content):
    old = git_meta(p)
    if old:
        if old["sha"] != blob_sha(content):
            raise RuntimeError("Immutable result differs: "+p+" — choose a NEW RUN_ID")
        return
    data = {"message": "Freeze Probe50 artifact: "+p.rsplit("/",1)[-1],
            "branch": "main", "content": base64.b64encode(content).decode()}
    r = gh.put(API_ROOT+p,json=data,timeout=180);r.raise_for_status()
    print("Saved",p)

raw = git_get(SOURCE_CSV)
assert raw is not None, "Candidate CSV missing"
pairs = pd.read_csv(io.BytesIO(raw), keep_default_na=False)
assert len(pairs)==50 and pairs.candidate_id.nunique()==50
assert (pairs.selection_status=="CANDIDATE_NOT_FROZEN").all()
names = sorted(set(pairs.target_x)|set(pairs.comparator_y))
print("Pairs:",len(pairs),"unique concept texts:",len(names))
PARAMS = {"run_id":RUN_ID,"source_csv_sha256":hashlib.sha256(raw).hexdigest(),
          "model":MODEL_ID,"revision":REVISION,"api_model":API_MODEL,
          "layers":LAYERS,"per_class":PER_CLASS,"shuffle_count":SHUFFLES,
          "probe_C":PROBE_C,"seed":SEED,"batch_size":BATCH_SIZE,
          "auc_min":AUC_MIN,"fdr_max":FDR_MAX,
          "styles":["structure","action","situation","function"],
          "examples_per_style_per_split":2,
          "text_generation_version":"v5_style_by_style_clean_notebook"}
config_raw = json.dumps(PARAMS,sort_keys=True,indent=2).encode()
git_put_immutable(f"{DEST}/config.json",config_raw)
git_put_immutable(f"{DEST}/candidate_pairs.csv",raw)
BASELINE=f"{ROOT}/results/20261124_probe50_independent_v1"
baseline_raw=git_get(f"{BASELINE}/pair_probe_scores.csv")
assert baseline_raw is not None, "v1 results are required for comparison"
baseline=pd.read_csv(io.BytesIO(baseline_raw)).set_index("candidate_id")
assert len(baseline)==50 and set(baseline.index)==set(pairs.candidate_id)
git_put_immutable(f"{DEST}/baseline_v1_pair_probe_scores.csv",baseline_raw)


Pairs: 50 unique concept texts: 82
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/config.json
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/candidate_pairs.csv
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/baseline_v1_pair_probe_scores.csv


## 1. Generate validated balanced descriptions

The API generates only **two sentences in one style per request**, and the notebook checks the count, word lengths, concept/comparator name leakage, duplicates and train/eval overlap. It retries only the specific failing style. Each conceptual class has eight train and eight held-out sentences (two of each type). All 82 concepts are covered. Source text is checkpointed per concept.

In [3]:
STYLES=("structure","action","situation","function")
STYLE_DETAILS={
    "structure":"Concrete physical appearance, observable components, composition and construction.",
    "action":"Characteristic motion, behaviour or operation in real life.",
    "situation":"A concrete location, environment, interaction or typical situation.",
    "function":"Its purpose, role, practical utility or natural consequence.",
}
def normalized(t):
    return " ".join(re.findall(r"[a-z0-9]+",unicodedata.normalize("NFKC",str(t)).lower()))
def has_label(t,label):
    # Word-level detection, case-insensitive. Further semantic alias review remains necessary.
    return bool(re.search(r"(?<![a-z])"+re.escape(label.lower())+r"(?![a-z])",t.lower()))
def similarity(a,b):
    return SequenceMatcher(None,normalized(a),normalized(b)).ratio()
forbidden={name:{name} for name in names}
for rec in pairs.itertuples():
    forbidden[rec.target_x].add(rec.comparator_y)
    forbidden[rec.comparator_y].add(rec.target_x)
SENTENCE_SCHEMA={"name":"two_descriptions","strict":True,"schema":{
    "type":"object",
    "properties":{"sentences":{"type":"array","items":{"type":"string"}}},
    "required":["sentences"],"additionalProperties":False}}
def check_two(name,arr,prior):
    if not isinstance(arr,list) or len(arr)!=2:
        return False,"Expected exactly two sentences; observed "+str(len(arr) if isinstance(arr,list) else type(arr).__name__)
    if any(not isinstance(s,str) or not 6<=len(s.split())<=40 for s in arr):
        return False,"Unexpected sentence lengths: "+repr([len(s.split()) if isinstance(s,str) else None for s in arr])
    if normalized(arr[0])==normalized(arr[1]) or similarity(arr[0],arr[1])>=.86:
        return False,"Two sentences duplicate each other"
    leaked=[x for x in arr for label in forbidden[name] if has_label(x,label)]
    if leaked:return False,"Target or comparator label present: "+repr(leaked[:1])
    if prior:
        peak=max(similarity(x,y) for x in arr for y in prior)
        if peak>=.70:return False,f"Existing text too similar (ratio={peak:.3f})"
    return True,"ok"
def make_two(name,style,split,prior,used):
    instruction=("Use distinct factual angles beyond the cited training examples." if split=="eval" else
                 "Use diverse elementary facts appropriate to this style.")
    base=(f"Target concept (never write its name): {name}\nStyle: {style}: {STYLE_DETAILS[style]}\n"
          f"Dataset part: {split}. {instruction}\n"
          "Produce exactly TWO separate, self-contained, true English sentences of 10-24 words each. "
          "Both sentences must describe the target (no unsupported claims), but never literally name it, "
          "its alternative names, or any of these forbidden words: "+repr(sorted(forbidden[name]))+
          "\nAvoid broad filler, numbered lists and sentence templates.")
    if prior:base+="\nTraining examples; use different facts:\n"+"\n".join(prior)
    reason=""
    for attempt in range(10):
        ask=base+(f"\nImportant: regenerate both sentences. Previous attempt failed: {reason}" if attempt else "")
        response=OPENAI.chat.completions.create(
            model=API_MODEL,temperature=.6,
            response_format={"type":"json_schema","json_schema":SENTENCE_SCHEMA},
            messages=[
                {"role":"system","content":"You create short natural fact descriptions as JSON and follow the two-sentence schema exactly."},
                {"role":"user","content":ask}])
        try:
            obj=json.loads(response.choices[0].message.content or "{}")
        except (ValueError,TypeError) as exc:
            reason="Invalid JSON response: "+str(exc)
            print(name,split,style,"retry",attempt+1,reason);continue
        sent=obj.get("sentences") if isinstance(obj,dict) else None
        ok,reason=check_two(name,sent,used)
        if ok:return sent
        print(name,split,style,"retry",attempt+1,reason)
    raise RuntimeError(f"Could not generate valid examples for {name}/{split}/{style}: {reason}. "
                       "Do not silently include malformed data. Use a new run ID if design changes.")
def flatten(bank):
    return [sent for style in STYLES for sent in bank[style]]
def check_bank(name,bank,other=None):
    if not isinstance(bank,dict) or set(bank)!=set(STYLES):
        return False,"Missing style key"
    collected=flatten(other) if other else []
    for style in STYLES:
        ok,why=check_two(name,bank[style],collected)
        if not ok:return False,style+": "+why
        collected.extend(bank[style])
    return True,"ok"
def create_bank(name,split,training=None):
    result={}
    used=flatten(training) if training else []
    for style in STYLES:
        sent=make_two(name,style,split,flatten(training) if training else [],used)
        result[style]=sent
        used+=sent
    return result
all_texts={}
for name in tqdm(names,desc="Generate balanced styles"):
    key=hashlib.sha256(name.encode()).hexdigest()[:12]
    remote=f"{DEST}/texts/concept_{key}.json"
    cached=git_get(remote)
    if cached is not None:
        d=json.loads(cached)
        assert d["concept"]==name and d["params"]==PARAMS
        a,why=check_bank(name,d["train"]);assert a,(name,"train",why)
        a,why=check_bank(name,d["eval"],d["train"]);assert a,(name,"eval",why)
    else:
        training=create_bank(name,"train")
        evaluation=create_bank(name,"eval",training)
        d={"concept":name,"params":PARAMS,"created_at":datetime.now(timezone.utc).isoformat(),
           "train":training,"eval":evaluation}
        git_put_immutable(remote,json.dumps(d,sort_keys=True,ensure_ascii=False,indent=2).encode())
    all_texts[name]=d
print("Generated/recovered",len(all_texts),"concept banks")


Generate balanced styles:   0%|          | 0/82 [00:00<?, ?it/s]

Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_8d3287c6fbb6.json
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_67a333356cdc.json
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_3a7bd3e2360a.json
armchair eval function retry 1 Existing text too similar (ratio=0.727)
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_5a948ffe5f0f.json
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_add1743eae5b.json
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_b493d48364af.json
bear train situation retry 1 Expected exactly two sentences; observed 1
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/texts/concept_bc98bb50e809.json
bed train situation retry 1 Expected exactly two sentences;

## 2. Audit style balance, lexical overlap and label leakage

The train/evaluation **style distributions are exactly equal** by construction. We check that every sentence is recorded with its style, and detect surface duplicates/near-duplicates as well as explicit X or paired-Y mentions. The style labels are never passed to Qwen. This audit cannot certify that two sentences do not express the same fact, that a model followed the intended style, or that their concepts are semantically equivalent across categories.

In [7]:
from difflib import SequenceMatcher

def calc_similarity(a, b):
    return SequenceMatcher(None, normalized(a), normalized(b)).ratio()

In [10]:
stim=[]
for name in names:
    data=all_texts[name]
    for split in ("train","eval"):
        for style in STYLES:
            for i,sentence in enumerate(data[split][style]):
                stim.append({"concept":name,"split":split,"style":style,"family":STYLES.index(style)*2+i,"user_prompt":sentence})
texts=pd.DataFrame(stim)
assert len(texts)==len(names)*2*PER_CLASS
assert (texts.groupby(["concept","split","style"]).size()==2).all()
audit=[]
for rec in pairs.itertuples():
    sub=texts[texts.concept.isin([rec.target_x,rec.comparator_y])]
    hits=[(x.concept,x.split,x.family,word) for x in sub.itertuples()
          for word in (rec.target_x,rec.comparator_y) if has_label(x.user_prompt,word)]
    audit.append({"candidate_id":rec.candidate_id,"target_x":rec.target_x,
                  "comparator_y":rec.comparator_y,"leaked_terms":repr(hits),
                  "n_leaks":len(hits)})
audit=pd.DataFrame(audit)
dup=[]
for name in names:
    aa=flatten(all_texts[name]["train"])
    bb=flatten(all_texts[name]["eval"])
    for i,a in enumerate(aa):
        for j,b in enumerate(bb):
            sim_score = calc_similarity(a, b)
            if sim_score >= 0.70:
                dup.append({
                    "concept": name,
                    "train_idx": i,
                    "eval_idx": j,
                    "similarity": sim_score,
                    "train": a,
                    "eval": b
                })
flags=pd.DataFrame(dup,columns=["concept","train_idx","eval_idx","similarity","train","eval"])
texts.to_csv(WORK/"balanced_descriptions.csv",index=False)
audit.to_csv(WORK/"lexical_audit.csv",index=False)
flags.to_csv(WORK/"overlap_flags.csv",index=False)
for name in ("balanced_descriptions.csv","lexical_audit.csv","overlap_flags.csv"):
    git_put_immutable(f"{DEST}/{name}",(WORK/name).read_bytes())
assert (audit.n_leaks==0).all(),"Name leakage detected; inspect lexical audit"
if len(flags) > 0:
    print(f"WARNING: {len(flags)} train/eval overlap(s) detected.")
    display(flags)
    print("Results require review; flagged concepts must not be treated as independently validated.")
print("Style count:",texts.groupby(["split","style"]).size().to_dict())
print("All 50 pair label-leak checks passed (surface-level only)")


,concept,train_idx,eval_idx,similarity,train,eval
0,fog,3,1,0.72103,"As the temperature drops at night, moisture co...","As the temperature drops, moisture condenses i..."


Results require review; flagged concepts must not be treated as independently validated.
Style count: {('eval', 'action'): 164, ('eval', 'function'): 164, ('eval', 'situation'): 164, ('eval', 'structure'): 164, ('train', 'action'): 164, ('train', 'function'): 164, ('train', 'situation'): 164, ('train', 'structure'): 164}
All 50 pair label-leak checks passed (surface-level only)


## 3. Qwen Hidden State extraction (same model and assistant-start position as v1)

**Only the explanation stimuli change.** The Qwen input remains the plain user message followed by the assistant generation prefix. Extract L22/23/24 at the last input token. Save immutable chunks; re-running a completed chunk loads the existing vectors. 82 concepts × 16 sentences = 1,312 unique inputs.

In [13]:
assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,revision=REVISION,torch_dtype=torch.float32,
    attn_implementation="eager").to("cuda").eval()
assert model.config.num_hidden_layers==24
@torch.inference_mode()
def extract(text):
    ids=tokenizer.apply_chat_template(
        [{"role":"user","content":text}],tokenize=True,
        add_generation_prompt=True,return_tensors="pt").to("cuda")
    hs=model(input_ids=ids,output_hidden_states=True,use_cache=False).hidden_states
    return np.stack([hs[l][0,-1].float().cpu().numpy() for l in LAYERS])

text_raw = (WORK / "balanced_descriptions.csv").read_bytes()
input_hash=hashlib.sha256(text_raw).hexdigest()
n=len(texts);n_batches=math.ceil(n/BATCH_SIZE)
features=[]
for b in tqdm(range(n_batches),desc="Qwen hidden states"):
    start,end=b*BATCH_SIZE,min(n,(b+1)*BATCH_SIZE)
    remote=f"{DEST}/features/chunk_{b:03d}.npz"
    saved=git_get(remote)
    if saved is None:
        chunk=np.stack([extract(t) for t in texts.user_prompt.iloc[start:end]])
        buf=io.BytesIO()
        np.savez_compressed(buf,features=chunk,begin=np.int64(start),end=np.int64(end),
                            input_hash=np.array(input_hash),layers=np.array(LAYERS))
        saved=buf.getvalue()
        git_put_immutable(remote,saved)
    with np.load(io.BytesIO(saved),allow_pickle=False) as f:
        assert int(f["begin"])==start and int(f["end"])==end
        assert str(f["input_hash"])==input_hash
        assert f["layers"].tolist()==LAYERS
        x=f["features"].copy()
        assert x.shape==(end-start,3,model.config.hidden_size) and np.isfinite(x).all()
        features.append(x)
vectors=np.concatenate(features)
assert len(vectors)==len(texts)
indices={(r.concept,r.split,r.family):i for i,r in enumerate(texts.itertuples())}
def values(name,split,layer_idx):
    return np.stack([vectors[indices[name,split,i],layer_idx] for i in range(PER_CLASS)])
print("Verified tensor",vectors.shape)


Qwen hidden states:   0%|          | 0/55 [00:00<?, ?it/s]

Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_000.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_001.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_002.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_003.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_004.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_005.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_006.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_007.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/features/chunk_008.npz
Saved experiments/2026-11/1124_probe_50pairs/results/20

## 4. Per-pair linear Probe: same fitting and screening as v1

- Frozen X/Y candidates, layers, `C=0.1`, seed, 8 training and 8 held-out examples per concept.
- Evaluate L22/L23/L24 AUC, L24 leave-one-training-fact-out minimum, and 499 class-balanced label permutations. BH correction across all 50.
- **Crucially:** FDR is reported as a diagnostic, not a sole operational requirement for whether the Probe can be investigated. We explicitly separate AUC≥0.80, uncorrected permutation p≤0.05, and FDR q≤0.05.
- Compare new balanced-text AUC against the fixed v1 results; these are **different explanation sets** so the delta is descriptive, not a randomized paired causal estimate of writing-type bias.
- A within-style held-out AUC uses only 2 sentences per class and must be treated as a **very noisy exploratory diagnostic**.

In [14]:
def fit_probe(a,b,labels=None):
    x=np.concatenate([a,b])
    labels=np.r_[np.ones(len(a),dtype=int),np.zeros(len(b),dtype=int)] if labels is None else labels
    scaler=StandardScaler().fit(x)
    clf=LogisticRegression(C=PROBE_C,solver="liblinear",max_iter=1000,random_state=SEED).fit(scaler.transform(x),labels)
    return scaler,clf
def auc(scaler,clf,a,b):
    score=clf.decision_function(scaler.transform(np.concatenate([a,b])))
    labels=np.r_[np.ones(len(a),dtype=int),np.zeros(len(b),dtype=int)]
    return float(roc_auc_score(labels,score))
def stable_seed(s):
    return int(hashlib.sha256((s+str(SEED)).encode()).hexdigest()[:8],16)

results=[]
for p in tqdm(pairs.itertuples(),total=len(pairs),desc="50 independent Probe screens"):
    row={"candidate_id":p.candidate_id,"category":p.category,"target_x":p.target_x,
         "comparator_y":p.comparator_y,"prior_evidence":p.prior_evidence}
    for li,l in enumerate(LAYERS):
        a,b=values(p.target_x,"train",li),values(p.comparator_y,"train",li)
        ea,eb=values(p.target_x,"eval",li),values(p.comparator_y,"eval",li)
        scaler,clf=fit_probe(a,b)
        row[f"L{l}_auc"]=auc(scaler,clf,ea,eb)
        if l==24:
            loo=[]
            for omit in range(PER_CLASS):
                keep=[j for j in range(PER_CLASS) if j!=omit]
                ss,cc=fit_probe(a[keep],b[keep])
                loo.append(auc(ss,cc,ea,eb))
            row["L24_loo_min"]=min(loo);row["L24_loo_mean"]=float(np.mean(loo))
            rng=np.random.default_rng(stable_seed(p.candidate_id))
            x=np.concatenate([a,b])
            y=np.r_[np.ones(PER_CLASS,dtype=int),np.zeros(PER_CLASS,dtype=int)]
            zx=scaler.transform(x)
            ze=scaler.transform(np.concatenate([ea,eb]))
            null=[]
            for _ in range(SHUFFLES):
                fake=rng.permutation(y)
                cc=LogisticRegression(C=PROBE_C,solver="liblinear",max_iter=1000,random_state=SEED).fit(zx,fake)
                null.append(roc_auc_score(y,cc.decision_function(ze)))
            row["p_shuffle_plus_one"]=(1+sum(v>=row["L24_auc"]-1e-12 for v in null))/(SHUFFLES+1)
            row["shuffle_mean_auc"]=float(np.mean(null))
    results.append(row)
scores=pd.DataFrame(results)
scores["fdr_q"]=multipletests(scores.p_shuffle_plus_one.to_numpy(),method="fdr_bh")[1]
scores["auc_gate"]=scores.L24_auc>=AUC_MIN
scores["fdr_gate"]=scores.fdr_q<=FDR_MAX
scores["screen_pass"]=scores.auc_gate & scores.fdr_gate
scores["selection_status"]=np.where(scores.screen_pass,"FDR_SCREEN_PASS_REVIEW_REQUIRED","NOT_FDR_CONFIRMED")
scores["new_validation_completed"]=True
scores["uncorrected_gate"]=(scores.p_shuffle_plus_one<=0.05)
scores["auc_and_uncorrected"]=scores.auc_gate & scores.uncorrected_gate
scores["L24_auc_v1"]=scores.candidate_id.map(baseline.L24_auc)
scores["L24_auc_delta_v2_minus_v1"]=scores.L24_auc-scores.L24_auc_v1
scores["auc_gate_v1"]=scores.candidate_id.map(baseline.auc_gate)
scores["auc_gate_flip"]=scores.auc_gate != scores.auc_gate_v1
for style in STYLES:
    idx=[STYLES.index(style)*2,STYLES.index(style)*2+1]
    out=[]
    for rec in pairs.itertuples():
        aa=values(rec.target_x,"train",2)
        bb=values(rec.comparator_y,"train",2)
        ee=values(rec.target_x,"eval",2)[idx]
        ff=values(rec.comparator_y,"eval",2)[idx]
        ss,cc=fit_probe(aa,bb)
        out.append(auc(ss,cc,ee,ff))
    scores[f"auc_style_{style}"]=out
scores.to_csv(WORK/"pair_probe_scores.csv",index=False)
display(scores[["candidate_id","target_x","comparator_y","L24_auc","L24_loo_min","fdr_q","screen_pass"]])
print("L24 AUC >= 0.80:",int(scores.auc_gate.sum()),"/",len(scores))
print("AUC and uncorrected permutation p<=0.05:",int(scores.auc_and_uncorrected.sum()))
print("AUC and BH q<=0.05:",int(scores.screen_pass.sum()))
print("Median v1/v2 L24 AUC:",float(scores.L24_auc_v1.median()),float(scores.L24_auc.median()))
print("AUC threshold crossings vs v1:",int(scores.auc_gate_flip.sum()))


50 independent Probe screens:   0%|          | 0/50 [00:00<?, ?it/s]

,candidate_id,target_x,comparator_y,L24_auc,L24_loo_min,fdr_q,screen_pass
0,C01,bicycle,submarine,1.000000,1.000000,0.011111,True
1,C02,boat,bus,1.000000,1.000000,0.017391,True
2,C03,bus,helicopter,0.984375,0.968750,0.011111,True
3,C04,motorcycle,helicopter,0.984375,0.953125,0.011111,True
4,C05,submarine,motorcycle,1.000000,1.000000,0.018519,True
5,C06,elephant,lion,1.000000,0.937500,0.011111,True
6,C07,lion,tiger,0.906250,0.843750,0.044118,True
7,C08,wolf,bear,0.921875,0.828125,0.011111,True
8,C09,giraffe,zebra,0.781250,0.750000,0.051351,False
9,C10,rabbit,hare,0.593750,0.500000,0.287500,False


L24 AUC >= 0.80: 35 / 50
AUC and uncorrected permutation p<=0.05: 33
AUC and BH q<=0.05: 33
Median v1/v2 L24 AUC: 0.7421875 0.90625
AUC threshold crossings vs v1: 20


## 5. Full results, side-by-side v1 comparison and interpretation

The results report all 50 candidate pairs and per-category differences. **Do not interpret v2 improvement as proof** that type imbalance alone caused v1's lower scores: factual content, API sampling, and sentence complexity also changed. This follow-up does not choose X/Y by Avoid/Neutral outcomes, and it must not silently overwrite the original v1 run.

All data and checkpoints are stored under a **new run ID**. Result statuses are screening diagnostics, not a final frozen concept inventory.

In [15]:
summary=[
"# Probe50 — balanced text-type follow-up v2",
"",
f"- Run ID: \`{RUN_ID}\`",
f"- Fixed 50 X/Y candidate pairs; unique concepts: {len(names)}",
f"- Inputs: {len(texts)} sentences (8 train + 8 held-out per concept)",
"- Balanced 4 styles per split: structure, action, situation, function (2 per style).",
f"- L24 AUC>=0.80: **{int(scores.auc_gate.sum())}/50** (v1: **{int(scores.auc_gate_v1.sum())}/50**)",
f"- L24 AUC>=0.80 and uncorrected p<=0.05: **{int(scores.auc_and_uncorrected.sum())}/50**",
f"- L24 AUC>=0.80 and BH q<=0.05: **{int(scores.screen_pass.sum())}/50**",
f"- Median L24 AUC, v1: **{scores.L24_auc_v1.median():.3f}**; v2: **{scores.L24_auc.median():.3f}**",
f"- Mean delta (v2-v1): **{scores.L24_auc_delta_v2_minus_v1.mean():+.3f}**; median delta: **{scores.L24_auc_delta_v2_minus_v1.median():+.3f}**",
f"- Pairs changing AUC>=0.80 classification: **{int(scores.auc_gate_flip.sum())}**",
"",
"## Scientific interpretation",
"",
"- v1-v2 differences are descriptive: new facts and API stochasticity also changed.",
"- Surface checks do not guarantee semantic fact independence or unbiased category/stylistic coverage.",
"- AI-generated statements have not been human fact-checked; Probe AUC is X/Y decodability, not suppression or absolute concept activation.",
"- Treat permutation p and BH q as exploratory diagnostics and report ALL 50 candidates.",
"",
"## By category",
"",
"| Category | v1 AUC≥0.8 | v2 AUC≥0.8 | v1 median AUC | v2 median AUC |",
"|---|---:|---:|---:|---:|",
]
for cat,g in scores.groupby("category",sort=True):
    summary.append(f"| {cat} | {int(g.auc_gate_v1.sum())} | {int(g.auc_gate.sum())} | {g.L24_auc_v1.median():.3f} | {g.L24_auc.median():.3f} |")
(WORK/"analysis_summary.md").write_text("\n".join(summary)+"\n",encoding="utf-8")
manifest={"status":"BALANCED_SCREEN_COMPLETED_HUMAN_REVIEW_PENDING",
          "timestamp":datetime.now(timezone.utc).isoformat(),"parameters":PARAMS,
          "input_sha256":input_hash,"baseline_sha256":hashlib.sha256(baseline_raw).hexdigest(),
          "n_pairs":len(scores),"n_names":len(names),"n_prompts":len(texts),
          "n_auc_pass":int(scores.auc_gate.sum()),
          "n_uncorrected_pass":int(scores.auc_and_uncorrected.sum()),
          "n_fdr_pass":int(scores.screen_pass.sum()),
          "limitations":["AI-authored statements; no independent human audit",
                         "AUC delta is not randomized causal effect of style balancing",
                         "AUC decodability is not absolute concept activation",
                         "FDR and p-values are screening diagnostics"]}
(WORK/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+"\n")
for filename in ("pair_probe_scores.csv","analysis_summary.md","manifest.json"):
    git_put_immutable(f"{DEST}/{filename}",(WORK/filename).read_bytes())
print((WORK/"analysis_summary.md").read_text())
print("Results:",DEST)


<>:4: SyntaxWarning: invalid escape sequence '\`'
<>:4: SyntaxWarning: invalid escape sequence '\`'
<>:4: SyntaxWarning: invalid escape sequence '\`'
<>:4: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipykernel_369/2563625060.py:4: SyntaxWarning: invalid escape sequence '\`'
  f"- Run ID: \`{RUN_ID}\`",
/tmp/ipykernel_369/2563625060.py:4: SyntaxWarning: invalid escape sequence '\`'
  f"- Run ID: \`{RUN_ID}\`",


Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/pair_probe_scores.csv
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/analysis_summary.md
Saved experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/manifest.json
# Probe50 — balanced text-type follow-up v2

- Run ID: \`20261124_probe50_balanced_v5\`
- Fixed 50 X/Y candidate pairs; unique concepts: 82
- Inputs: 1312 sentences (8 train + 8 held-out per concept)
- Balanced 4 styles per split: structure, action, situation, function (2 per style).
- L24 AUC>=0.80: **35/50** (v1: **17/50**)
- L24 AUC>=0.80 and uncorrected p<=0.05: **33/50**
- L24 AUC>=0.80 and BH q<=0.05: **33/50**
- Median L24 AUC, v1: **0.742**; v2: **0.906**
- Mean delta (v2-v1): **+0.162**; median delta: **+0.164**
- Pairs changing AUC>=0.80 classification: **20**

## Scientific interpretation

- v1-v2 differences are descriptive: new facts and API stochasticity also changed.
- Surfa